## Motivation and model

At each moment of individual ball control, a player either releases a pass or keeps carrying the ball. Action selection estimates the probability of these two observed decisions from the current tracking state and match context. [Stalpaert (2022)](https://belgianfootball.s3.eu-central-1.amazonaws.com/s3fs-public/rbfa/docs/pdf/rbfa_knowledge_centre/studies/RBFA_Study_30_Counter_Risk.pdf) studies this binary pass–carry choice, and [Fernández, Bornn and Cervone (2021)](https://link.springer.com/article/10.1007/s10994-021-05989-6) treat a related problem with three actions (shooting, passing, carrying) inside a possession-value framework. PRESS covers outfield players and the pass–carry decision; shooting is not modelled.

Let $s$ denote the current game state and $A$ the chosen action. The target is

$$
\pi(s)=P(A=\mathrm{pass}\mid s),\qquad
P(A=\mathrm{carry}\mid s)=1-\pi(s).
$$

We model action selection as a **binary classification problem** and estimate $\pi(s)$ with an XGBoost classifier ([Chen and Guestrin, 2016](https://arxiv.org/abs/1603.02754)). The classifier takes a vector of features that summarizes the game state and returns the probability of passing. The features are described first, followed by the likelihood used to train the model.

These are probabilities of *choosing* an action, not of executing it successfully. The model describes observed behaviour across the sampled decisions; it is neither an optimal decision rule nor a measure of individual skill. Before choosing, a player weighs the risk of carrying, the space available and the prospects of a pass. PRESS represents these considerations through features computed from tracking data and from its separately fitted execution and destination-choice models.

## Model features

The model uses 30 features, 28 numerical and two categorical, in four groups. Each group answers one question about the decision. Under each feature, the text first says what it means in plain terms and then how it enters the model.

| Group | Question | Count |
|---|---|---:|
| Carrier and accessible space | How well can the player keep carrying? | 6 |
| Spatial and tactical configuration | Where does the carry lead, and how are the teams organized there? | 8 |
| Passing opportunities | What would a pass look like, and how would it end? | 11 |
| Possession and match context | How did control start, and what is the match situation? | 5 |

### 1. Carrier and accessible space (6 features)

These features describe the player and the room he has to keep moving with the ball.

- **x, y position.** Where the carrier is on the pitch, in the team's attacking orientation: deep in his own half or near the opponent's goal, central or wide.
- **Speed.** How fast the carrier is moving: running with the ball, or almost standing.
- **Velocity angle.** The direction of movement relative to the attacking axis: forward, sideways or backward. It is missing when the player is effectively stationary. Velocity comes from the existing centred seven-frame tracking smoother; its three-frame future support makes the current implementation suitable for offline analysis.
- **Carry-success probability $D(s)$.** The chance that the player still has the ball one second from now if he keeps carrying:

$$
D(s)=1-P(\mathrm{loss\ within\ 1\,s}\mid A=\mathrm{carry},s).
$$

  A high value means carrying is safe in the current situation. It comes from the separate [carry-risk model](../01_ball_drive_risk/01_ball_drive_risk.ipynb) and measures execution risk, not the probability of choosing a carry.
- **BCOS area.** Ball carrier open space (BCOS) is the space the carrier can navigate without interference from an opponent, a concept introduced for association football by [Bandara et al. (2025)](https://doi.org/10.1007/s42979-025-03815-7). In PRESS, the [BCOS model](../extras/bcos/10_bcos.ipynb) gives, for each point around the carrier, the probability that he arrives there before an opponent can intercept; integrating this field over a 40 m surrounding domain gives a probability-weighted accessible area in $\mathrm{m}^2$. A large value means room to run, a small one means he is boxed in.

$D(s)$ tells whether control would survive the next second, and BCOS area tells how much space is available. Both describe the carry option from the same current state.

### 2. Spatial and tactical configuration (8 features)

These features describe where the carry would take the player and how the two teams are arranged around that place.

Six of them are evaluated at a **projected position**: where the carrier would be one second from now if he kept his current velocity,

$$
\widehat x_B(1)=x_B(0)+v_B(0)\,(1\,\mathrm{s}).
$$

This is a hypothetical point, not an observed future position. The features ask what the situation looks like where the carry is heading.

- **Own-team xT.** The expected threat of the projected position for the carrier's team, that is, how much a possession at that spot tends to lead towards a goal. A high value means the carry heads into a dangerous attacking zone. xT was introduced by [Karun Singh (2018)](https://karun.in/blog/expected-threat.html); its construction in PRESS is described in the [xT vignette](../extras/expected_threat/11.expected_threat.ipynb).
- **Opponent xT.** The same quantity from the opponent's point of view. It indicates how dangerous that spot would be for the carrier's team if possession were lost there.
- **Nearest own-team line index** and **nearest opposition line index.** These are the pressure lines of [Fernández, Bornn and Cervone (2021)](https://link.springer.com/article/10.1007/s10994-021-05989-6). In PRESS, the current outfield longitudinal positions are clustered into three lines with complete linkage, excluding padding and the historical extreme-x goalkeeper proxy. The index says which line is closest to the projected position, for the carrier's own team and for the opposition. Opponent indices follow the carrier's attacking axis, whereas own-team indices keep the opposite-axis ordering used in pass selection.
- **Signed distance to the nearest own-team line** and **to the nearest opposition line.** How far that line is from the projected position along the attacking direction. A positive value means the line is ahead of the point and a negative value means it is behind. A small positive distance to the opposition line means the carrier is about to meet it.

The other two features use the carrier's *current* position.

- **Teammates ahead** and **opponents ahead.** How many players of each team are further up the pitch than the carrier, along the attacking direction. They show how many teammates are in front (potential forward options) and how many opponents are still to be beaten. For a set of players $G$ and a point $r$,

$$
N_G(r)=\sum_{k\in G}\mathbf 1\{d\,x_k>d\,r_x\},\qquad d\in\{-1,+1\},
$$

  where $d$ is the carrier team's attacking direction for both rosters. The carrier and padding are excluded and tracked goalkeepers are kept. The count is taken over the whole pitch width and is not a count of players in a passing corridor.

### 3. Passing opportunities and possible outcomes (11 features)

These features summarize what a pass would look like from the current state. A pass has two uncertain parts: *where* it would be directed and *whether it would succeed*. Two separately fitted PRESS components provide one probability for each part, for every candidate destination $j$ on the native one-metre pitch grid:

$$
q_j(s)=P(J=j\mid A=\mathrm{pass},s),\qquad
p_j(s)=P(\mathrm{completion}\mid J=j,A=\mathrm{pass},s).
$$

The first, $q_j$, says how likely a passer in this situation is to target cell $j$ and comes from the [pass-destination model](../03_transition_model/03_pass_selection.ipynb). The second, $p_j$, says how likely a pass to that cell is to arrive and comes from the [pass-probability model](../02_pass_probability/02_pass_probability.ipynb). The action model uses the flat destination model with its fitted validation temperature and isotonic-calibrated completion probabilities. Both are evaluated from the current game state, without the destination or outcome of the actual future action.

Multiplying them splits every possible pass into two branches, a completed pass and a failed pass:

$$
w_j^+=q_jp_j,\qquad w_j^-=q_j(1-p_j).
$$

The weight $w_j^+$ is the chance that the pass goes to cell $j$ and succeeds, and $w_j^-$ is the chance that it goes there and fails.

- **Expected pass success.** The overall chance that a pass would succeed, averaged over the likely destinations:

$$
P(\mathrm{completion}\mid A=\mathrm{pass},s)=\sum_jq_jp_j=\sum_jw_j^+.
$$

  A high value means passing is easy right now.

The other ten features take a spatial property $F(j)$ of each destination and add it up separately over the two branches:

$$
F^+(s)=\sum_jw_j^+F(j),\qquad
F^-(s)=\sum_jw_j^-F(j).
$$

Read $F^+$ as "the value of $F$ where completed passes would land" and $F^-$ as "the value of $F$ where failed passes would be aimed", each scaled by how likely that outcome is. They are *joint* weighted sums and are not divided by the overall probability of success or failure, so a branch that is unlikely contributes a small number whatever its destinations look like. Failure is always represented at the intended destination, and no observed interception or recovery point is used.

- **Expected threat (2 features).** $\sum_jw_j^+X_{own}(j)$ is the own-team threat of the places a completed pass would reach: how much a successful pass would advance the attack. $\sum_jw_j^-X_{opp}(j)$ is the opponent's threat at the places where a failed pass would be aimed: how dangerous it would be to lose the ball there.
- **Pressure-line structure (4 features).** The nearest own-team and opponent line indices, as defined in the previous group but evaluated at each destination from the current player positions, are aggregated once with $w_j^+$ and once with $w_j^-$:

$$
\sum_jw_j^+L_{own}(j),\quad \sum_jw_j^-L_{own}(j),\quad
\sum_jw_j^+L_{opp}(j),\quad \sum_jw_j^-L_{opp}(j).
$$

  They say which lines a successful pass would reach and in which part of the structure a failed pass would end.
- **Player positioning (4 features).** The ahead-count function $N_G$ is evaluated at every candidate destination for teammates and opponents, with the carrier team's attacking direction, and aggregated the same way:

$$
\sum_jw_j^+N_{own}(j),\quad \sum_jw_j^-N_{own}(j),\quad
\sum_jw_j^+N_{opp}(j),\quad \sum_jw_j^-N_{opp}(j).
$$

  They say how many teammates and opponents would still be ahead of the likely destinations. A backward destination can have more players ahead of it than a forward one.

Together, the eleven features give the classifier the likelihood, location and consequences of passing, without choosing a destination itself.

### 4. Possession and match context (5 features)

These features describe how the current individual control began and the match situation.

- **Start type** (categorical). The event that gave the player the ball: a reception, interception, recovery or restart.
- **Previous action** (categorical). What the player did immediately before. It keeps the start context until the player completes a full one-second carry, and from then on it records `ball_drive`.
- **Match minute.** Time elapsed in the match.
- **Score difference.** Carrier-team goals minus opponent goals at that frame.
- **Completed carries.** The number of complete one-second carry intervals already performed during the current individual possession, in other words how long the player has already been on the ball. It is a history summary available at decision time and not the eventual possession duration. A pass released after 0.4 s has zero completed carries and keeps its start context; a pass released after 2.4 s has two completed carries and `ball_drive` as previous action.

The two categorical variables keep their original labels, including `unknown`, and the encoded vocabulary is fitted on training data. The [possession and checkpoint vignette](../setup/possession_and_checkpoint_setup.ipynb) describes the underlying segmentation.

## Likelihood and learning

The model is trained on observed decisions. Each decision $i$ has two ingredients: the feature vector $z_i$ described above, which summarizes the game state at that moment, and its outcome $y_i$, equal to 1 if the player passed and 0 if he carried.

XGBoost learns the probability of passing from these pairs. It builds an additive ensemble of regression trees $f_\theta$, in which each new tree corrects the errors left by the previous ones. The ensemble returns a score for every decision, and a logistic function turns that score into the probability of passing:

$$
\pi_i=P(y_i=1\mid z_i)=\frac{1}{1+\exp[-f_\theta(z_i)]}.
$$

Training chooses the parameters $\theta$ that minimize the regularized Bernoulli negative log-likelihood, which penalizes confident predictions that disagree with the observed action:

$$
\mathcal L(\theta)=-\sum_{i=1}^{n}\left[y_i\log\pi_i+(1-y_i)\log(1-\pi_i)\right]+\Omega(\theta),
$$

where $\Omega(\theta)$ penalizes tree complexity and leaf weights to limit overfitting. The [action-selection benchmark](../../benchmarks/04_action_selection.qmd) reports the hyperparameter search, match-wise validation and predictive performance.

## Feature importance and interpretation

The contribution of each feature to the fitted model is measured with XGBoost split gain. The figure reports the total gain attributed to each conceptual feature, with one-hot encoded levels grouped under their original categorical input. These values describe how the trained ensemble uses its inputs; they do not measure causal effects or percentages of predictive accuracy.

![Feature importance for the fitted action-selection model.](../../figures/feature_importance_action_selection.png)

The carry-success probability $D(s)$ (`drive_success_1s`) is the most influential feature: it accounts for **22.09%** of total gain in the production model and ranks first in all five evaluation folds. Carrier speed follows at **8.00%**, so the player's immediate capacity to advance with the ball is central to the pass–carry decision. The passing-option summaries also contribute substantially: the own-team line index weighted by successful passing outcomes (`team_line_pass_success`) accounts for **6.85%**, and the opposition line index weighted by failed outcomes (`opp_line_pass_failure`) for **5.32%**. BCOS contributes **4.10%** and the previous-action context **4.04%**.

Together, these results show that the classifier combines the feasibility of continuing the carry with the spatial consequences of the available passing opportunities. Correlated features can share gain among themselves, and split gain depends on the opportunities a feature offers for tree partitioning. The full fold-wise analysis is documented in the [feature-importance analysis](../../data/benchmarks/feature_importance/README.md).

## Application to a game state

Brisbane FC–Perth Glory: N. Acosta chooses a pass at checkpoint 54295, and J. Hingert receives it at frame 54309. The first plot reports the model's probability of choosing a pass at that checkpoint.


In [3]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
from PRESS import PRESS

opendata = Path(os.environ.get("SKILLCORNER_OPENDATA", Path.cwd().parent / "opendata"))
match_dir = opendata / "data/processed/2026-10-15_training_v5_outfield/matches/1925299"
press = PRESS.from_skillcorner(
    match_dir, actions_path=match_dir / "1925299_action_selection_base.parquet"
)
frame_id = 54295

from PRESS.model import PassSelectionModel
from PRESS.extras import BCOS

xT_grid = PassSelectionModel.pretrained().xt_grid
area = float(BCOS(press.tracking_data).compute_bcos(frame_id)["BCOS area"])
result = press.compute_frame(
    frame_id, xT_grid, bcos_value=area,
    match_clock_min=85.075, score_difference_prefix=-1.0,
)

from PRESS.visualization import plot_action_selection

plot_action_selection(press, frame_id)
plt.savefig("figures/vignette_04_action_pitch.png", dpi=170)
plt.savefig("figures/vignette_04_action_pitch.svg")
plt.close()


![The real decision checkpoint, with predicted P(pass) above the pitch.](../../figures/vignette_04_action_pitch.png)

In [5]:
from PRESS.visualization import plot_frame

reception_frame = 54309
plot_frame(press, reception_frame, title="Matched reception: J. Hingert · frame 54309")
plt.savefig("figures/vignette_04_action_reception.png", dpi=170)
plt.savefig("figures/vignette_04_action_reception.svg")
plt.close()


![The matched reception of the same pass.](../../figures/vignette_04_action_reception.png)

## Example: outcome-weighted passing features

The following synthetic three-destination example illustrates how the joint weights are used to construct the four line and four ahead-count aggregates. It does not use fitted model predictions.


In [8]:
import numpy as np
import pandas as pd

q = np.array([0.2, 0.5, 0.3])
p = np.array([0.9, 0.6, 0.8])
success, failure = q * p, q * (1 - p)
spatial = {
    "team_line": np.array([1, 2, 3]),
    "opponent_line": np.array([3, 2, 1]),
    "teammates_ahead": np.array([7, 4, 2]),
    "opponents_ahead": np.array([8, 5, 3]),
}
joint = pd.DataFrame({
    "success_weighted": {name: float(success @ value) for name, value in spatial.items()},
    "failure_weighted": {name: float(failure @ value) for name, value in spatial.items()},
})
assert np.isclose(success.sum(), 0.72)
for name, value in spatial.items():
    assert np.isclose(joint.loc[name].sum(), q @ value)
print(f"Expected pass success: {success.sum():.2f}")
print(joint.to_string())


Expected pass success: 0.72
                 success_weighted  failure_weighted
team_line                    1.50              0.60
opponent_line                1.38              0.52
teammates_ahead              2.94              1.06
opponents_ahead              3.66              1.34


The [benchmark](../../benchmarks/04_action_selection.qmd) contains the cohort definition, match-disjoint validation, XGBoost tuning, held-out metrics and published comparison. The examples above illustrate the fitted model and its feature construction; they are not benchmark evaluations.
